# Final Python Notebook 3: Ensemble Classifier & Regression Decision Trees
## 5DATA002W.2 Machine Learning & Data Mining — Case Study A (Ensemble) & Case Study B (Regression)
**Author:** [Your Name Here]  
**Code Peer Review conducted by:** [Peer Reviewer Name]  
**Date of Peer Review:** [Date]

## Section 1: Import Libraries
Code block leveraged and reused from: Seminar Session — Ensemble and Regression Lab

In [ ]:
# Import pandas for data manipulation
import pandas as pd

# Import numpy for numerical operations
import numpy as np

# Import visualisation libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Import preprocessing utilities
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Import data splitting utility
from sklearn.model_selection import train_test_split

# Import classification algorithms for ensemble base learners
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier

# Import Voting Ensemble classifier
from sklearn.ensemble import VotingClassifier

# Import Decision Tree Regressor for regression modelling
from sklearn.tree import DecisionTreeRegressor, export_text, plot_tree

# Import classification evaluation metrics
from sklearn.metrics import (confusion_matrix, classification_report,
                              accuracy_score, recall_score, precision_score,
                              f1_score, roc_auc_score, roc_curve, ConfusionMatrixDisplay)

# Import regression evaluation metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Suppress warnings for clean output
import warnings
warnings.filterwarnings('ignore')

## Section 2: Load and Preprocess Data
Code block leveraged and reused from: Seminar Session — Data Preparation Lab

In [ ]:
# Load the loan approval dataset
df = pd.read_csv('loan_approval_data__3_.csv')

# Drop the id column — no predictive value
df_clean = df.drop(columns=['id'])

# Remove age outlier rows (age > 100)
df_clean = df_clean[df_clean['age'] <= 100]

# Remove employment length outlier rows (> 50 years)
df_clean = df_clean[df_clean['emplyment_length'] <= 50]

# Set negative loan interest rates to NaN before imputation
df_clean.loc[df_clean['loan_interest_rate'] < 0, 'loan_interest_rate'] = np.nan

# Set negative max_allowed_loan values to NaN
df_clean.loc[df_clean['max_allowed_loan'] < 0, 'max_allowed_loan'] = np.nan

# Impute missing age values with median
df_clean['age'] = df_clean['age'].fillna(df_clean['age'].median())

# Impute missing loan_interest_rate with median
df_clean['loan_interest_rate'] = df_clean['loan_interest_rate'].fillna(df_clean['loan_interest_rate'].median())

# Impute missing payment_default_on_file with mode
df_clean['payment_default_on_file'] = df_clean['payment_default_on_file'].fillna(df_clean['payment_default_on_file'].mode()[0])

# Impute missing max_allowed_loan with median
df_clean['max_allowed_loan'] = df_clean['max_allowed_loan'].fillna(df_clean['max_allowed_loan'].median())

# Encode categorical variables using LabelEncoder
le_home    = LabelEncoder()
le_intent  = LabelEncoder()
le_default = LabelEncoder()

df_clean['home_ownership_enc'] = le_home.fit_transform(df_clean['home_ownership'])
df_clean['loan_intent_enc']    = le_intent.fit_transform(df_clean['loan_intent'])
df_clean['payment_default_enc'] = le_default.fit_transform(df_clean['payment_default_on_file'])

print('Preprocessing complete. Shape:', df_clean.shape)

## Section 3: Prepare Classification Data and Build Base Learners (NB & KNN)
Code block leveraged and reused from: Seminar Session — Classification Lab

In [ ]:
# Define classification feature set
clf_features = ['age', 'income', 'home_ownership_enc', 'emplyment_length',
                'loan_intent_enc', 'loan_amount', 'loan_interest_rate',
                'loan_income_ratio', 'payment_default_enc', 'credit_history_length']

# Create input feature matrix and target vector for classification
X = df_clean[clf_features]
y = df_clean['loan_approval_status']

# Split data into 80% training and 20% test sets with stratification for class balance
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Apply standard scaling to training and test sets
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print('Classification split done. Train:', X_train.shape, '| Test:', X_test.shape)

In [ ]:
# Declare Naive Bayes base learner
nb_base = GaussianNB()

# Declare K-Nearest Neighbours base learner with k=5
knn_base = KNeighborsClassifier(n_neighbors=5)

# Fit NB base learner on training data
nb_base.fit(X_train_scaled, y_train)

# Fit KNN base learner on training data
knn_base.fit(X_train_scaled, y_train)

print('Base learners trained.')

In [ ]:
# Generate predictions for NB base learner
y_pred_nb   = nb_base.predict(X_test_scaled)
y_prob_nb   = nb_base.predict_proba(X_test_scaled)[:, 1]

# Generate predictions for KNN base learner
y_pred_knn  = knn_base.predict(X_test_scaled)
y_prob_knn  = knn_base.predict_proba(X_test_scaled)[:, 1]

# Helper function to display confusion matrix
def plot_cm(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['Approved(0)', 'Rejected(1)'])
    fig, ax = plt.subplots(figsize=(5, 4))
    disp.plot(ax=ax, cmap='Blues')
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

# Display confusion matrix for NB base learner
plot_cm(y_test, y_pred_nb, 'Confusion Matrix — NB (Base Learner)')

In [ ]:
# Print classification report for NB base learner
print('=== NB Base Learner Classification Report ===')
print(classification_report(y_test, y_pred_nb, target_names=['Approved(0)', 'Rejected(1)']))

In [ ]:
# Display confusion matrix for KNN base learner
plot_cm(y_test, y_pred_knn, 'Confusion Matrix — KNN (Base Learner)')

In [ ]:
# Print classification report for KNN base learner
print('=== KNN (k=5) Base Learner Classification Report ===')
print(classification_report(y_test, y_pred_knn, target_names=['Approved(0)', 'Rejected(1)']))

In [ ]:
# Plot AUC-ROC curves for NB and KNN base learners
fig, ax = plt.subplots(figsize=(7, 5))
for y_prob, name in [(y_prob_nb, 'NB'), (y_prob_knn, 'KNN (k=5)')]:
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    ax.plot(fpr, tpr, label=f'{name} (AUC = {auc:.4f})')
ax.plot([0, 1], [0, 1], 'k--', label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('AUC-ROC Curves — NB and KNN Base Learners')
ax.legend()
plt.tight_layout()
plt.show()

## Section 4: Voting Ensemble Classifier (NB + KNN)
Code block leveraged and reused from: Seminar Session — Ensemble Lab

In [ ]:
# Import the VotingClassifier and re-declare base learners for ensemble construction
from sklearn.ensemble import VotingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier

# Declare NB base learner for use in ensemble
nb_ens = GaussianNB()

# Declare KNN base learner (k=5) for use in ensemble
knn_ens = KNeighborsClassifier(n_neighbors=5)

# Build soft-voting ensemble combining NB and KNN
# soft voting uses predicted probabilities, which is more informative than hard voting
ensemble = VotingClassifier(
    estimators=[('nb', nb_ens), ('knn', knn_ens)],
    voting='soft'
)

# Fit the ensemble learner on the training data
ensemble.fit(X_train_scaled, y_train)

# Generate ensemble predictions and probabilities on test set
y_pred_ens = ensemble.predict(X_test_scaled)
y_prob_ens = ensemble.predict_proba(X_test_scaled)[:, 1]

print('Ensemble learner (NB + KNN) trained and predictions generated.')

In [ ]:
# Display confusion matrix for the voting ensemble learner
plot_cm(y_test, y_pred_ens, 'Confusion Matrix — Voting Ensemble (NB + KNN)')

In [ ]:
# Print classification report for the voting ensemble learner
print('=== Ensemble Learner (NB + KNN) Classification Report ===')
print(classification_report(y_test, y_pred_ens, target_names=['Approved(0)', 'Rejected(1)']))

In [ ]:
# Compare metrics: NB, KNN, and Ensemble
print(f"{'Model':<20} {'Accuracy':>10} {'Recall':>10} {'Precision':>10} {'F1':>10} {'AUC':>10}")
print('-' * 70)
for name, y_pred, y_prob in [
    ('NB (Base)', y_pred_nb, y_prob_nb),
    ('KNN k=5 (Base)', y_pred_knn, y_prob_knn),
    ('Ensemble (NB+KNN)', y_pred_ens, y_prob_ens)
]:
    print(f"{name:<20} {accuracy_score(y_test,y_pred):>10.4f} {recall_score(y_test,y_pred):>10.4f} "
          f"{precision_score(y_test,y_pred):>10.4f} {f1_score(y_test,y_pred):>10.4f} "
          f"{roc_auc_score(y_test,y_prob):>10.4f}")

---
# Case Study B: Regression — Predicting Maximum Loan Amount

## Section 5: Prepare Regression Dataset
Code block leveraged and reused from: Seminar Session — Regression Lab

In [ ]:
# Filter dataset to approved clients only (loan_approval_status = 0, i.e. Credit Acceptance = Yes)
# and positive max_allowed_loan values (the regression target)
reg_df = df_clean[(df_clean['loan_approval_status'] == 0) & (df_clean['max_allowed_loan'] > 0)]

# Define regression feature set
reg_features = ['age', 'income', 'home_ownership_enc', 'emplyment_length',
                'loan_intent_enc', 'loan_amount', 'loan_interest_rate',
                'loan_income_ratio', 'payment_default_enc', 'credit_history_length']

# Create regression input feature matrix and target vector
X_reg = reg_df[reg_features]
y_reg = reg_df['max_allowed_loan']

print('Regression dataset — Feature names:', X_reg.columns.tolist())
print('Regression dataset — Shape (X):', X_reg.shape)
print('Regression dataset — Shape (y):', y_reg.shape)

In [ ]:
# Split regression data into 80% training and 20% test sets
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42)

print('Train size:', X_train_r.shape, '| Test size:', X_test_r.shape)

## Section 6: Build Regression Decision Trees — DT-1 (Fully Grown) and DT-2 (Pruned)
Code block leveraged and reused from: Seminar Session — Regression Lab

In [ ]:
# Import DecisionTreeRegressor for building regression models
from sklearn.tree import DecisionTreeRegressor

# Declare DT-1: Fully grown decision tree regressor (no depth limit)
dt1 = DecisionTreeRegressor(random_state=42)

# Fit DT-1 on the regression training data
dt1.fit(X_train_r, y_train_r)

# Declare DT-2: Pruned decision tree regressor limited to maximum depth of 4
dt2 = DecisionTreeRegressor(max_depth=4, random_state=42)

# Fit DT-2 on the regression training data
dt2.fit(X_train_r, y_train_r)

print('DT-1 (Fully Grown) — Depth:', dt1.get_depth(), '| Leaves:', dt1.get_n_leaves())
print('DT-2 (Pruned, depth=4) — Depth:', dt2.get_depth(), '| Leaves:', dt2.get_n_leaves())

## Section 7: Visualise Regression Decision Trees
Code block leveraged and reused from: Seminar Session — Regression Lab

In [ ]:
# Visualise DT-1 (Fully Grown) — limit display to depth 3 for readability
plt.figure(figsize=(24, 10))
plot_tree(dt1, feature_names=reg_features, filled=True, rounded=True,
          max_depth=3, fontsize=8)
plt.title('DT-1: Fully Grown Decision Tree Regressor (first 3 levels shown)')
plt.tight_layout()
plt.show()

In [ ]:
# Visualise DT-2 (Pruned to depth=4) — full tree shown as it is compact
plt.figure(figsize=(24, 10))
plot_tree(dt2, feature_names=reg_features, filled=True, rounded=True,
          fontsize=9)
plt.title('DT-2: Pruned Decision Tree Regressor (max_depth=4)')
plt.tight_layout()
plt.show()

## Section 8: Evaluate Regression Models — MSE, MAE, R²
Code block leveraged and reused from: Seminar Session — Regression Lab

In [ ]:
# Generate test predictions for DT-1 and DT-2
y_pred_dt1 = dt1.predict(X_test_r)
y_pred_dt2 = dt2.predict(X_test_r)

# Compute and print regression evaluation metrics for both models
print(f"{'Metric':<12} {'DT-1 (Fully Grown)':>22} {'DT-2 (Pruned)':>18}")
print('-' * 55)
print(f"{'MSE':<12} {mean_squared_error(y_test_r, y_pred_dt1):>22,.2f} {mean_squared_error(y_test_r, y_pred_dt2):>18,.2f}")
print(f"{'MAE':<12} {mean_absolute_error(y_test_r, y_pred_dt1):>22,.2f} {mean_absolute_error(y_test_r, y_pred_dt2):>18,.2f}")
print(f"{'R-Square':<12} {r2_score(y_test_r, y_pred_dt1):>22.4f} {r2_score(y_test_r, y_pred_dt2):>18.4f}")

## Section 9: Predict Maximum Loan Amount for Client 60256
Code block leveraged and reused from: Seminar Session — Regression Lab

In [ ]:
# Define client 60256 attributes as per the coursework specification
client_60256 = pd.DataFrame([{
    'age': 56,
    'income': 57000,
    'home_ownership_enc': le_home.transform(['RENT'])[0],     # RENT encoded value
    'emplyment_length': 15,
    'loan_intent_enc': le_intent.transform(['MEDICAL'])[0],   # MEDICAL encoded value
    'loan_amount': 25700,
    'loan_interest_rate': 23.0,
    'loan_income_ratio': 0.10,
    'payment_default_enc': le_default.transform(['N'])[0],    # N (No default) encoded value
    'credit_history_length': 35
}])

# Predict the Maximum Loan Amount for client 60256 using the selected best model (DT-2)
pred_max_loan_dt2 = dt2.predict(client_60256)[0]

print(f'DT-2 Predicted Maximum Loan Amount for Client 60256: £{pred_max_loan_dt2:,.2f}')

In [ ]:
# Show the decision path taken in DT-2 for client 60256 to interpret the prediction
# Export the tree structure as a text rule set for interpretation
tree_rules = export_text(dt2, feature_names=reg_features)
print('DT-2 Decision Tree Rules:')
print(tree_rules)

In [ ]:
# Show the decision node path for client 60256 in DT-2
node_indicator = dt2.decision_path(client_60256)
leaf_id = dt2.apply(client_60256)

# Identify which nodes the sample passes through
node_ids = node_indicator.indices[node_indicator.indptr[0]:node_indicator.indptr[1]]

print(f'Client 60256 passes through {len(node_ids)} nodes in DT-2.')
print(f'Nodes visited: {node_ids}')
print(f'Leaf node ID: {leaf_id[0]}')